### [DELETE] Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.

This notebook will:
1. **Purge all soft-deleted resources first** (APIM, Key Vault, Cognitive Services, Log Analytics, ML Workspaces) — these block redeployment with the same names
2. **Delete the resource group** (async, runs in background)
3. **Verify** cleanup is complete (run the SWEEP cell)

> **Important:** Always run this notebook before redeploying if a previous deployment was canceled or deleted. Soft-deleted resources linger for 7-90 days and will cause `ServiceAlreadyExistsInSoftDeletedState` errors.

In [ ]:
import os, sys, subprocess, json, time

def run_cmd(cmd, print_output=True):
    """Run Azure CLI command and return result"""
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = (proc.stdout or "").strip()
    error = (proc.stderr or "").strip()
    
    if print_output and output:
        print(output)
    if error and print_output:
        print(error)
    
    return {"success": proc.returncode == 0, "output": output, "error": error}

deployment_name = "azureml-integration-with-agents"
resource_group = f"lab-{deployment_name}"

# Get subscription ID (needed for several purge operations)
sub_info = run_cmd('az account show --query id -o tsv', print_output=False)
subscription_id = sub_info['output'].strip() if sub_info['success'] else ""
if not subscription_id:
    raise RuntimeError("Unable to get subscription ID. Run 'az login' first.")

print(f"[DELETE] Cleanup for: {resource_group} (sub: {subscription_id})\n")

# ─── Step 1: Purge ALL soft-deleted resources FIRST (before RG deletion) ───
# These block redeployment with the same names regardless of RG state.

# 1a) Purge soft-deleted APIM services
print("[1/6] Purging soft-deleted API Management services...")
apim_list = run_cmd('az apim deletedservice list -o json', print_output=False)
if apim_list['success'] and apim_list['output']:
    try:
        deleted_apims = json.loads(apim_list['output'])
        for apim in (deleted_apims or []):
            apim_name = apim.get('name', '')
            apim_location = apim.get('location', '')
            if apim_name and apim_location:
                print(f"  Purging APIM: {apim_name} ({apim_location})")
                run_cmd(f'az apim deletedservice purge --service-name {apim_name} --location "{apim_location}"', print_output=False)
                print(f"    [OK] Purge initiated: {apim_name}")
    except json.JSONDecodeError:
        pass
if not (apim_list['success'] and apim_list['output'] and json.loads(apim_list['output'])):
    print("  [OK] No soft-deleted APIM services found")

# 1b) Purge soft-deleted Key Vaults
print("\n[2/6] Purging soft-deleted Key Vaults...")
kv_list = run_cmd('az keyvault list-deleted --query "[].{name:name, location:properties.location}" -o json', print_output=False)
if kv_list['success'] and kv_list['output']:
    try:
        deleted_kvs = json.loads(kv_list['output'])
        for kv in (deleted_kvs or []):
            kv_name = kv.get('name', '')
            kv_location = kv.get('location', '')
            if kv_name and kv_location:
                print(f"  Purging Key Vault: {kv_name} ({kv_location})")
                run_cmd(f'az keyvault purge --name {kv_name} --location "{kv_location}"', print_output=False)
                print(f"    [OK] Purge initiated: {kv_name}")
    except json.JSONDecodeError:
        pass
if not (kv_list['success'] and kv_list['output'] and json.loads(kv_list['output'])):
    print("  [OK] No soft-deleted Key Vaults found")

# 1c) Purge soft-deleted Cognitive Services (AI Services / Foundry)
print("\n[3/6] Purging soft-deleted Cognitive Services...")
cog_list = run_cmd('az cognitiveservices account list-deleted -o json', print_output=False)
if cog_list['success'] and cog_list['output']:
    try:
        deleted_cogs = json.loads(cog_list['output'])
        for cog in (deleted_cogs or []):
            cog_name = cog.get('name', '')
            cog_location = cog.get('location', '')
            cog_rg = cog.get('resourceGroup', resource_group)
            if cog_name and cog_location:
                print(f"  Purging Cognitive Service: {cog_name} ({cog_location})")
                run_cmd(
                    f'az cognitiveservices account purge --name {cog_name} '
                    f'--resource-group {cog_rg} --location "{cog_location}"',
                    print_output=False
                )
                print(f"    [OK] Purge initiated: {cog_name}")
    except json.JSONDecodeError:
        pass
if not (cog_list['success'] and cog_list['output'] and json.loads(cog_list['output'])):
    print("  [OK] No soft-deleted Cognitive Services found")

# 1d) Purge soft-deleted Log Analytics workspaces
# NOTE: When a RG is deleted before the workspace, the resourceGroup field becomes null.
# We must match by BOTH resourceGroup name AND null resourceGroup (orphaned workspaces from this lab).
print("\n[4/6] Purging soft-deleted Log Analytics workspaces...")
law_list = run_cmd(
    f'az rest --method GET '
    f'--url "https://management.azure.com/subscriptions/{subscription_id}/providers/Microsoft.OperationalInsights/deletedWorkspaces" '
    f'--url-parameters api-version=2021-12-01-preview '
    f'-o json',
    print_output=False
)
if law_list['success'] and law_list['output']:
    try:
        law_data = json.loads(law_list['output'])
        all_deleted = law_data.get('value', []) if isinstance(law_data, dict) else []
        # Match workspaces that belonged to our RG OR have null resourceGroup (orphaned after RG delete)
        deleted_laws = [
            ws for ws in all_deleted
            if ws.get('properties', {}).get('resourceGroupName') == resource_group
            or (ws.get('properties', {}).get('resourceGroupName') is None
                and 'q2xpi7j3qsht2' in ws.get('name', ''))
            or ws.get('properties', {}).get('resourceGroupName') == resource_group
        ]
        for ws in deleted_laws:
            law_name = ws.get('name', '')
            if law_name:
                print(f"  Force-deleting Log Analytics: {law_name}")
                # Ensure RG exists for the recover command
                rg_check_law = run_cmd(f'az group exists --name {resource_group}', print_output=False)
                if rg_check_law['output'].strip().lower() != 'true':
                    run_cmd(f'az group create --name {resource_group} --location swedencentral -o none', print_output=False)
                # Recover then force-delete to permanently purge
                run_cmd(
                    f'az monitor log-analytics workspace recover '
                    f'--workspace-name {law_name} --resource-group {resource_group}',
                    print_output=False
                )
                run_cmd(
                    f'az monitor log-analytics workspace delete '
                    f'--workspace-name {law_name} --resource-group {resource_group} --yes --force',
                    print_output=False
                )
                print(f"    [OK] Force-deleted: {law_name}")
    except json.JSONDecodeError:
        pass
if not (law_list['success'] and law_list['output']):
    print("  [OK] No soft-deleted Log Analytics workspaces found")
elif not deleted_laws:
    print("  [OK] No soft-deleted Log Analytics workspaces found for this lab")

# 1e) Purge soft-deleted ML Workspaces
# ML workspaces don't have a list-deleted CLI command — we must use the REST API.
# The RG may already be deleted, so we recreate it temporarily if needed.
print("\n[5/6] Purging soft-deleted ML Workspaces...")

# Ensure resource group exists (needed by the purge API even if workspace is soft-deleted)
rg_check = run_cmd(f'az group exists --name {resource_group}', print_output=False)
rg_existed = rg_check['output'].strip().lower() == 'true'
if not rg_existed:
    run_cmd(f'az group create --name {resource_group} --location swedencentral -o none', print_output=False)

# Try to purge the workspace using the known naming pattern: aml-<resourceSuffix>
# The resourceSuffix is derived from uniqueString(subscription, resourceGroup)
# We can discover it from any previous deployment or by trying the known name.
ml_purge_url = (
    f"https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
    f"/providers/Microsoft.MachineLearningServices/workspaces"
)

# List workspaces (including soft-deleted) via REST
ml_list = run_cmd(
    f'az rest --method GET --url "{ml_purge_url}" '
    f'--url-parameters api-version=2024-04-01 includeSoftDeleted=true -o json',
    print_output=False
)

purged_any = False
if ml_list['success'] and ml_list['output']:
    try:
        ml_data = json.loads(ml_list['output'])
        ml_workspaces = ml_data.get('value', []) if isinstance(ml_data, dict) else []
        for ws in ml_workspaces:
            ws_name = ws.get('name', '')
            ws_state = ws.get('properties', {}).get('provisioningState', '')
            if ws_name and ws_state.lower() in ('softdeleted', 'deleted', ''):
                print(f"  Purging ML Workspace: {ws_name} (state: {ws_state or 'soft-deleted'})")
                purge_result = run_cmd(
                    f'az rest --method DELETE '
                    f'--url "{ml_purge_url}/{ws_name}" '
                    f'--url-parameters api-version=2024-04-01 forceToPurge=true',
                    print_output=False
                )
                if purge_result['success']:
                    print(f"    [OK] Purged: {ws_name}")
                else:
                    print(f"    [WARN] Purge may have failed: {purge_result['error'][:100]}")
                purged_any = True
    except json.JSONDecodeError:
        pass

if not purged_any:
    print("  [OK] No soft-deleted ML Workspaces found")

# Clean up the temporary resource group if we created it just for purging
if not rg_existed:
    run_cmd(f'az group delete --name {resource_group} --yes --no-wait', print_output=False)

# ─── Step 2: Delete the resource group ───
print(f"\n[6/6] Deleting resource group: {resource_group}...")
check = run_cmd(f'az group exists --name {resource_group}', print_output=False)
rg_exists = check['output'].strip().lower() == 'true'

if rg_exists:
    result = run_cmd(f'az group delete --name {resource_group} --yes --no-wait', print_output=False)
    if result['success']:
        print(f"  [OK] Resource group deletion initiated (running in background)")
    else:
        print(f"  [FAILED] {result['error']}")
else:
    print(f"  [OK] Resource group already deleted or doesn't exist")

print(f"\n{'='*60}")
print(f"[DONE] Cleanup complete.")
print(f"  - Soft-deleted resource purges may take 5-15 minutes to propagate")
print(f"  - Resource group deletion takes 2-5 minutes in background")
print(f"  - Run the SWEEP cell below to verify everything is clean")

### [SWEEP] Verify cleanup and check for remaining soft-deleted resources

If the resource group was already deleted (e.g., via the Azure Portal), soft-deleted resources may still linger and block redeployment with the same names. This cell sweeps for and verifies them.

In [ ]:
# Verify final cleanup status
print(f"[STATUS] Verifying cleanup status...\n")

# Check if resource group still exists
final_check = run_cmd(f'az group exists --name {resource_group}', print_output=False)
rg_exists = final_check['output'].strip().lower() == 'true'

if not rg_exists:
    print(f"[OK] Resource group '{resource_group}' deleted")
else:
    print(f"[WARN] Resource group still exists (deletion may be in progress)")

# Check for remaining soft-deleted resources
print(f"\n[RESOURCES] Soft-deleted resources status:\n")

def _list_deleted(cmd: str) -> list:
    result = run_cmd(cmd, print_output=False)
    if not result['success'] or not result['output']:
        return []
    try:
        data = json.loads(result['output'])
        return data if isinstance(data, list) else []
    except Exception:
        return []

# APIM
apim_deleted = _list_deleted('az apim deletedservice list -o json')
if apim_deleted:
    print(f"  [WARN] {len(apim_deleted)} soft-deleted APIM service(s):")
    for a in apim_deleted:
        print(f"         - {a.get('name')} ({a.get('location')})")
else:
    print(f"  [OK] No soft-deleted APIM services")

# Key Vault
kv_deleted = _list_deleted('az keyvault list-deleted -o json')
if kv_deleted:
    print(f"  [WARN] {len(kv_deleted)} soft-deleted Key Vault(s):")
    for k in kv_deleted:
        name = k.get('name', k.get('properties', {}).get('vaultId', ''))
        print(f"         - {name}")
else:
    print(f"  [OK] No soft-deleted Key Vaults")

# Cognitive Services
cog_deleted = _list_deleted('az cognitiveservices account list-deleted -o json')
if cog_deleted:
    print(f"  [WARN] {len(cog_deleted)} soft-deleted Cognitive Services account(s):")
    for c in cog_deleted:
        print(f"         - {c.get('name')} ({c.get('location')})")
else:
    print(f"  [OK] No soft-deleted Cognitive Services")

# Log Analytics — check ALL deleted workspaces (resourceGroup becomes null after RG deletion)
law_all = _list_deleted(
    f'az rest --method GET '
    f'--url "https://management.azure.com/subscriptions/{subscription_id}/providers/Microsoft.OperationalInsights/deletedWorkspaces" '
    f'--url-parameters api-version=2021-12-01-preview '
    f'-o json'
)
# The REST API returns {value: [...]} — _list_deleted won't parse it as a list, so handle dict
if not law_all:
    law_check = run_cmd(
        f'az rest --method GET '
        f'--url "https://management.azure.com/subscriptions/{subscription_id}/providers/Microsoft.OperationalInsights/deletedWorkspaces" '
        f'--url-parameters api-version=2021-12-01-preview '
        f'--query "value[]" -o json',
        print_output=False
    )
    law_all = json.loads(law_check['output']) if law_check['success'] and law_check['output'] else []
law_deleted = [
    ws for ws in law_all
    if ws.get('properties', {}).get('resourceGroupName') == resource_group
    or ws.get('properties', {}).get('resourceGroupName') is None
]
if law_deleted:
    print(f"  [WARN] {len(law_deleted)} soft-deleted Log Analytics workspace(s):")
    for l in law_deleted:
        print(f"         - {l.get('name')}")
else:
    print(f"  [OK] No soft-deleted Log Analytics workspaces")

print(f"\n{'='*60}")
if any([apim_deleted, kv_deleted, cog_deleted, law_deleted]):
    print(f"[ACTION] Some resources still soft-deleted. Wait 5-10 min and re-run this cell.")
    print(f"         If they persist, re-run the DELETE cell above to purge them.")
else:
    print(f"[ALL CLEAR] No soft-deleted resources found. Safe to redeploy.")